# 03 - Preprocessing

This notebook cleans the raw data, encodes categorical features and the target, splits the data into train/test sets, and applies SMOTE to the training set to correct class imbalance. It saves the resulting arrays as pickles for `04_train_model.ipynb`.

## 1. Imports and load data

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

df = pd.read_pickle('../data/raw_df.pkl')
df.shape

## 2. Drop the customer identifier

`customerID` is a unique identifier with no predictive value, so we drop it before modeling.

In [ ]:
df = df.drop(columns=['customerID'])

## 3. Fix `TotalCharges`

`TotalCharges` is read as an `object` column because a small number of rows (new customers with 0 tenure) contain blank strings instead of numbers. We coerce it to numeric, turning those blanks into NaN, and then fill the resulting NaNs with 0 — the correct value for a brand-new customer who has not yet been billed.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
n_missing = df['TotalCharges'].isnull().sum()
print(f'Rows with unparseable TotalCharges: {n_missing}')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

## 4. Handle any other missing values

A defensive check: if any other column has missing values, drop those rows (there normally are none in this dataset besides `TotalCharges`).

In [ ]:
missing_counts = df.isnull().sum()
print(missing_counts[missing_counts > 0])
df = df.dropna()
df.shape

## 5. Encode the target

Map `Churn` from Yes/No to 1/0.

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Churn'].value_counts()

## 6. Encode binary categorical columns

Several columns only take two values. We label-encode these as 0/1 directly (rather than one-hot encoding, which would be redundant for a binary column).

In [ ]:
binary_maps = {
    'gender': {'Female': 0, 'Male': 1},
    'Partner': {'No': 0, 'Yes': 1},
    'Dependents': {'No': 0, 'Yes': 1},
    'PhoneService': {'No': 0, 'Yes': 1},
    'PaperlessBilling': {'No': 0, 'Yes': 1},
}

for col, mapping in binary_maps.items():
    df[col] = df[col].map(mapping)

df[list(binary_maps.keys())].head()

## 7. One-hot encode multi-category columns

The remaining categorical columns take more than two values (some including a 'No internet/phone service' option), so we one-hot encode them with `pd.get_dummies`, dropping the first level of each to avoid the dummy-variable trap.

In [ ]:
multi_cat_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod',
]

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)
df.head()

## 8. Split into features (X) and target (y)

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']
print(f'X shape: {X.shape}, y shape: {y.shape}')

## 9. Stratified train/test split

We hold out 20% of the data for testing, stratifying on `y` so both splits preserve the original churn/no-churn ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 10. Scale numeric features

We fit a `StandardScaler` on the numeric columns of the **training set only**, then apply the same transform to the test set. Fitting on train-only avoids leaking test-set statistics into training.

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')
print('Saved scaler to ../models/scaler.pkl')

## 11. Save the feature column list

The one-hot encoding step produces a fixed set of feature columns. We save this exact column list so the Streamlit app can align a single new customer's encoded features to match — filling in 0 for any dummy category not present in that one row.

In [ ]:
feature_columns = X_train.columns.tolist()
joblib.dump(feature_columns, '../models/feature_columns.pkl')
print(f'Saved {len(feature_columns)} feature columns to ../models/feature_columns.pkl')

## 12. Apply SMOTE to the training set only

As seen in the EDA notebook, churn is imbalanced (~27% positive class). We apply **SMOTE** (Synthetic Minority Over-sampling Technique) to synthesize new minority-class (churned) examples **only on the training set**. Applying SMOTE before the split, or to the test set, would leak synthetic, non-independent samples derived from test points into training/evaluation and give an overly optimistic — and invalid — estimate of model performance. The test set must stay untouched and reflect the real-world class distribution.

In [ ]:
print('Before SMOTE:', y_train.value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('After SMOTE:', y_train_res.value_counts().to_dict())

## 13. Save the processed splits

We persist the (resampled) training set and the original, untouched test set as pickles for the training notebook.

In [ ]:
X_train_res.to_pickle('../data/X_train.pkl')
X_test.to_pickle('../data/X_test.pkl')
y_train_res.to_pickle('../data/y_train.pkl')
y_test.to_pickle('../data/y_test.pkl')

print('Saved X_train, X_test, y_train, y_test to ../data/')